# Part 4: Synthetic Test Dataset Generation (Golden Dataset Creation)
Building an evaluation suite (like the ones we set up in Parts 2 and 3) requires a "Golden Dataset"—a collection of realistic user questions paired with their source context chunks and ground-truth answers.

Manually writing hundreds of test cases for an enterprise document repository is tedious, expensive, and prone to human bias (developers write questions they think users will ask, missing edge cases). Synthetic Test Dataset Generation solves this by using LLMs to automatically ingest your domain documents and spin up diverse, production-grade test sets.

## 1. How Synthetic Test Generation Works (The Ragas Generator Pattern)
Instead of asking an LLM to blindly invent questions, modern test generators (like Ragas' TestsetGenerator) use your actual ingested document chunks as seed data, applying Evol-Instruct methodologies to build diverse question types:

**Simple Queries:** Straightforward factual questions answerable by a single text chunk (e.g., "What is the primary product of TechCorp Europe?").

**Reasoning Queries:** Questions that require multi-step logical deduction from the text.

**Multi-Context Queries:** Complex queries that force the retriever to pull and stitch together information from two or more distinct chunks across different documents.

**Conditional Queries:** Questions containing specific conditional constraints (e.g., "If Berlin port strikes occur, how does it affect distribution?").

## 2. Implementation Code (generate_synthetic_testset.py)
Here is a complete, production-ready Python script that takes local markdown or text documents, extracts chunks, and automatically generates a synthetic test dataset using Ragas and OpenAI:

In [ ]:
"""
generate_synthetic_testset.py
Demonstrates automatically generating a golden evaluation test dataset 
from proprietary enterprise documents using the Ragas TestsetGenerator.
"""

import os
from langchain_community.document_loaders import DirectoryLoader
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings
from ragas.testset import TestsetGenerator

def generate_golden_dataset():
    # 1. Load domain documents from your repository folder
    docs_path = "./sample_docs/" # Update to point to your document directory
    
    # Fallback simulation if directory doesn't exist yet for testing
    if not os.path.exists(docs_path):
        os.makedirs(docs_path, exist_ok=True)
        with open(os.path.join(docs_path, "company_policy.md"), "w") as f:
            f.write("# TechCorp Global Operations\nTechCorp Europe faced supply chain bottlenecks in Q3 due to port strikes in Berlin. DataStream Logistics handles tier-1 routing solutions across Germany.")

    print(f"Loading documents from: {docs_path}")
    loader = DirectoryLoader(docs_path, glob="**/*.md")
    documents = loader.load()

    if not documents:
        print("No documents found. Please add markdown files to the target directory.")
        return

    # 2. Configure the Generator LLM (e.g., GPT-4o) and Embedder
    generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o", temperature=0.7))
    generator_embeddings = RagasOpenAIEmbeddings(model="text-embedding-3-small")

    # 3. Initialize the Ragas TestsetGenerator
    generator = TestsetGenerator(
        llm=generator_llm,
        embedding_model=generator_embeddings
    )

    print("Generating synthetic test dataset (this may take a minute)...")

    # 4. Generate the test dataset specifying test size distribution
    # Testset generation creates questions, target contexts, and ground truths
    testset = generator.generate_with_langchain_docs(
        documents=documents,
        test_size=5,  # Number of synthetic samples to generate
        distributions={
            "simple": 0.5,
            "reasoning": 0.25,
            "multi_context": 0.25
        }
    )

    # 5. Convert to pandas DataFrame and save as a Golden Dataset CSV
    df_testset = testset.to_pandas()
    output_filename = "golden_eval_dataset.csv"
    df_testset.to_csv(output_filename, index=False)
    
    print(f"\nSuccessfully generated golden evaluation dataset!")
    print(f"Saved to: {output_filename}")
    print("\nSample Generated Test Cases:")
    print(df_testset[["question", "evolution_type", "ground_truth"]].head())

if __name__ == "__main__":
    if "OPENAI_API_KEY" not in os.environ:
        print("Error: OPENAI_API_KEY environment variable is required.")
    else:
        generate_golden_dataset()

## 3. Best Practices for Synthetic Test Data Generation

**Curate Your Seed Corpus:** The quality of your synthetic test set depends entirely on the quality of your source documents. Garbage in means hallucinated, overly complex questions out.

**Version Control Your Golden Datasets:** Save your generated CSVs (e.g., golden_eval_dataset_v1.csv) directly inside your repository. Whenever you update your chunking strategy or prompt templates, run your evaluation pipeline against this fixed dataset to ensure fair, apples-to-apples performance comparisons.

**Human-in-the-Loop Review:** While automated test generation saves hours of manual work, have domain experts review a subset of the generated questions to filter out unnatural phrasing or impossible edge cases.